In [5]:
'''
🔍 Affinity Propagation (AP) 개요
    - 특징:
        - KMeans처럼 클러스터 수(k)를 미리 지정하지 않아도 됨
        - 데이터 포인트들 사이의 유사도를 기반으로 “대표점(Exemplar)”을 자동으로 선택해 클러스터를 형성
    - 장점:
        - 클러스터 수를 자동으로 결정 → 사용자가 k를 지정하기 어려운 경우 유용
        - 대표 문서를 자연스럽게 뽑아낼 수 있음 (Exemplar = 클러스터 중심 역할)
    - 단점:
        - 계산량이 많아 대규모 데이터셋에서는 느릴 수 있음
        - 파라미터(preference, damping)에 따라 결과가 크게 달라질 수 있음
'''

'\n🔍 Affinity Propagation (AP) 개요\n    - 특징:\n        - KMeans처럼 클러스터 수(k)를 미리 지정하지 않아도 됨\n        - 데이터 포인트들 사이의 유사도를 기반으로 “대표점(Exemplar)”을 자동으로 선택해 클러스터를 형성\n    - 장점:\n        - 클러스터 수를 자동으로 결정 → 사용자가 k를 지정하기 어려운 경우 유용\n        - 대표 문서를 자연스럽게 뽑아낼 수 있음 (Exemplar = 클러스터 중심 역할)\n    - 단점:\n        - 계산량이 많아 대규모 데이터셋에서는 느릴 수 있음\n        - 파라미터(preference, damping)에 따라 결과가 크게 달라질 수 있음\n'

In [6]:
import os
import sys
import glob

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from pycaret.clustering import setup, create_model, assign_model, models, pull
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from datetime import datetime
import warnings
import time

from utils import preprocessing

In [7]:
warnings.filterwarnings('ignore')
DATA_PATH = '../data'
OUTPUT_DIR = '../results'
IMAGE_DIR = '../images'
    
document_df = preprocessing.get_default_data()

📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [8]:
# 모델 비교와 자동 선택 통합 코드

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, AffinityPropagation
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ---------------------------
# 공통 유틸
# ---------------------------
def compute_metrics(X_emb, labels):
    """라벨로부터 세 가지 군집 지표 계산. 클러스터가 2개 이상일 때만 유효."""
    unique = set(labels)
    if len(unique) <= 1:
        return {"silhouette": None, "davies_bouldin": None, "calinski": None}
    sil = silhouette_score(X_emb, labels)
    db  = davies_bouldin_score(X_emb, labels)
    ch  = calinski_harabasz_score(X_emb, labels)
    return {"silhouette": sil, "davies_bouldin": db, "calinski": ch}

def compute_metrics_excluding_noise(X_emb, labels):
    """DBSCAN 전용: noise(-1) 제외하고 지표 계산."""
    mask = labels != -1
    if mask.sum() == 0:
        return {"silhouette": None, "davies_bouldin": None, "calinski": None}
    sub_labels = labels[mask]
    sub_X = X_emb[mask]
    unique = set(sub_labels)
    if len(unique) <= 1:
        return {"silhouette": None, "davies_bouldin": None, "calinski": None}
    sil = silhouette_score(sub_X, sub_labels)
    db  = davies_bouldin_score(sub_X, sub_labels)
    ch  = calinski_harabasz_score(sub_X, sub_labels)
    return {"silhouette": sil, "davies_bouldin": db, "calinski": ch}

def normalize(values, higher_is_better=True):
    """0-1 정규화. None은 제외하고 처리."""
    arr = np.array(values, dtype=float)
    vmin, vmax = np.nanmin(arr), np.nanmax(arr)
    if np.isclose(vmax - vmin, 0):
        return np.ones_like(arr) * 0.5  # 모두 같은 값이면 중립
    if higher_is_better:
        return (arr - vmin) / (vmax - vmin)
    else:
        return (vmax - arr) / (vmax - vmin)

# ---------------------------
# 모델별 최적 탐색
# ---------------------------
def best_kmeans(X_emb, k_range=(2, 12), random_state=42):
    best = {"k": None, "labels": None, "metrics": None}
    records = []
    for k in range(k_range[0], k_range[1] + 1):
        km = KMeans(n_clusters=k, random_state=random_state)
        labels = km.fit_predict(X_emb)
        m = compute_metrics(X_emb, labels)
        records.append((k, m))
    # 실루엣 최고 기준으로 선택 (None 제외)
    valid = [(k, m) for k, m in records if m["silhouette"] is not None]
    if not valid:
        return best
    best_k, best_m = max(valid, key=lambda x: x[1]["silhouette"])
    km = KMeans(n_clusters=best_k, random_state=random_state)
    best_labels = km.fit_predict(X_emb)
    return {"k": best_k, "labels": best_labels, "metrics": compute_metrics(X_emb, best_labels)}

def best_agglomerative(X_emb, k_range=(2, 12)):
    best = {"k": None, "labels": None, "metrics": None}
    records = []
    for k in range(k_range[0], k_range[1] + 1):
        agg = AgglomerativeClustering(n_clusters=k)
        labels = agg.fit_predict(X_emb)
        m = compute_metrics(X_emb, labels)
        records.append((k, m))
    valid = [(k, m) for k, m in records if m["silhouette"] is not None]
    if not valid:
        return best
    best_k, best_m = max(valid, key=lambda x: x[1]["silhouette"])
    agg = AgglomerativeClustering(n_clusters=best_k)
    best_labels = agg.fit_predict(X_emb)
    return {"k": best_k, "labels": best_labels, "metrics": compute_metrics(X_emb, best_labels)}

def best_dbscan(X_emb, eps_list=(0.5, 0.8, 1.0, 1.5, 2.0), min_samples_list=(3,5,10)):
    best = {"params": None, "labels": None, "metrics": None}
    candidates = []
    for eps in eps_list:
        for ms in min_samples_list:
            dbs = DBSCAN(eps=eps, min_samples=ms)
            labels = dbs.fit_predict(X_emb)
            m = compute_metrics_excluding_noise(X_emb, labels)
            candidates.append(((eps, ms), labels, m))
    valid = [c for c in candidates if c[2]["silhouette"] is not None]
    if not valid:
        return best
    eps_ms, labels, m = max(valid, key=lambda x: x[2]["silhouette"])
    return {"params": {"eps": eps_ms[0], "min_samples": eps_ms[1]}, "labels": labels, "metrics": m}

def best_affinity_propagation(X_emb, damping=0.9, preference=None, random_state=42):
    ap = AffinityPropagation(damping=damping, preference=preference, random_state=random_state)
    labels = ap.fit_predict(X_emb)
    m = compute_metrics(X_emb, labels)
    return {"params": {"damping": damping, "preference": preference}, "labels": labels, "metrics": m}

# ---------------------------
# 통합 파이프라인
# ---------------------------
def evaluate_and_select_best_model(document_df, text_column='processed_text',
                                   max_features=5000, svd_dim=50, k_range=(2,12),
                                   eps_list=(0.5, 0.8, 1.0, 1.5, 2.0), min_samples_list=(3,5,10),
                                   random_state=42):
    # 1) TF-IDF
    texts = document_df[text_column].astype(str)
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
    X = vectorizer.fit_transform(texts)

    # 2) SVD 차원 축소
    svd = TruncatedSVD(n_components=svd_dim, random_state=random_state)
    X_reduced = svd.fit_transform(X)

    # 3) 모델별 최적 파라미터/성능 계산
    kmeans_res = best_kmeans(X_reduced, k_range=k_range, random_state=random_state)
    agglom_res = best_agglomerative(X_reduced, k_range=k_range)
    dbscan_res = best_dbscan(X_reduced, eps_list=eps_list, min_samples_list=min_samples_list)
    ap_res     = best_affinity_propagation(X_reduced, damping=0.9, preference=None, random_state=random_state)

    # 4) 모델별 지표 취합
    models = {
        "KMeans": kmeans_res["metrics"],
        "Agglomerative": agglom_res["metrics"],
        "DBSCAN": dbscan_res["metrics"],
        "AffinityPropagation": ap_res["metrics"]
    }

    # None 제거를 고려한 정규화 벡터 구성
    sil_vals = [models[m]["silhouette"] if models[m]["silhouette"] is not None else np.nan for m in models]
    db_vals  = [models[m]["davies_bouldin"] if models[m]["davies_bouldin"] is not None else np.nan for m in models]
    ch_vals  = [models[m]["calinski"] if models[m]["calinski"] is not None else np.nan for m in models]

    sil_norm = normalize(sil_vals, higher_is_better=True)
    db_norm  = normalize(db_vals,  higher_is_better=False)
    ch_norm  = normalize(ch_vals,  higher_is_better=True)

    combined = sil_norm + db_norm + ch_norm
    # NaN을 최소 점수로 처리
    combined = np.nan_to_num(combined, nan=0.0)

    model_names = list(models.keys())
    best_idx = int(np.argmax(combined))
    best_model_name = model_names[best_idx]

    # 5) 최적 모델 라벨 및 세부 파라미터 반환
    if best_model_name == "KMeans":
        best_info = kmeans_res
    elif best_model_name == "Agglomerative":
        best_info = agglom_res
    elif best_model_name == "DBSCAN":
        best_info = dbscan_res
    else:
        best_info = ap_res

    labels = best_info["labels"]
    document_df["Cluster"] = labels

    return {
        "X_reduced": X_reduced,
        "models_metrics": models,
        "normalized_scores": {
            "silhouette": dict(zip(model_names, sil_norm)),
            "davies_bouldin": dict(zip(model_names, db_norm)),
            "calinski": dict(zip(model_names, ch_norm)),
            "combined": dict(zip(model_names, combined))
        },
        "best_model": {
            "name": best_model_name,
            "details": best_info
        },
        "document_df": document_df
    }

In [ ]:
result = evaluate_and_select_best_model(
    document_df,
    text_column='processed_text',
    max_features=5000,
    svd_dim=50,
    k_range=(2, 12),
    eps_list=(0.5, 0.8, 1.0, 1.5, 2.0),
    min_samples_list=(3, 5, 10),
    random_state=42
)

print("=== 모델별 원점수 ===")
for name, m in result["models_metrics"].items():
    print(f"{name}: Sil={m['silhouette']}, DBI={m['davies_bouldin']}, CH={m['calinski']}")

print("\n=== 정규화 점수 및 종합 점수 ===")
for name, score in result["normalized_scores"]["combined"].items():
    s = result["normalized_scores"]["silhouette"][name]
    d = result["normalized_scores"]["davies_bouldin"][name]
    c = result["normalized_scores"]["calinski"][name]
    print(f"{name}: combined={score:.3f} (sil={s:.3f}, db={d:.3f}, ch={c:.3f})")

print("\n=== 최적 모델 ===")
best = result["best_model"]
print(f"Best model: {best['name']}")
print(f"Details: {best['details']}")

result = '''
- 현재 데이터셋에서는 DBSCAN이 최적 모델로 선택
- DBSCAN 모델을 기반으로 클러스터별 대표 문서
'''

=== 모델별 원점수 ===
KMeans: Sil=0.1658829046099021, DBI=1.7454585553424442, CH=2.862871159729943
Agglomerative: Sil=0.23442160751309588, DBI=1.7707452836647264, CH=3.3441679010548317
DBSCAN: Sil=0.5839214101301996, DBI=0.5097132147441809, CH=15.319400361497125
AffinityPropagation: Sil=0.19038361986124813, DBI=1.2249076710362743, CH=2.6972786343287174

=== 정규화 점수 및 종합 점수 ===
KMeans: combined=0.033 (sil=0.000, db=0.020, ch=0.013)
Agglomerative: combined=0.215 (sil=0.164, db=0.000, ch=0.051)
DBSCAN: combined=3.000 (sil=1.000, db=1.000, ch=1.000)
AffinityPropagation: combined=0.491 (sil=0.059, db=0.433, ch=0.000)

=== 최적 모델 ===
Best model: DBSCAN
Details: {'params': {'eps': 0.5, 'min_samples': 3}, 'labels': array([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0,  0,  0, -1,
       -1, -1, -1, -1,  1,  1,  1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
      dtype=int64), 'metrics': {'silhouette': 0.5839214101301996,